# Nano-Positioning System (NPS): forward models in IMP.bff

This **synthetic, executable** introduction distinguishes three models rather
than treating all FRET observables as distances:

1. Direct FRET with an explicit average transition-dipole orientation and
   depolarization (the NPS network model) [1, 2].
2. Transfer anisotropy and a multi-pair, **normalized Gaussian likelihood**
   in the Fast-NPS convention [2, 3].
3. Position-cloud distance convolution, which is **not** the same as a FRET
   calculation at the cloud's mean position [3, 4].

`nps_phase3.ipynb` continues with clouds, position priors, and a position-only
sampler. No numbers below are measured data. Coordinates and Förster radii are
in ångströms (Å); efficiency is dimensionless. The example uses the
non-negative NPS depolarization branch $q=\sqrt{r_{\mathrm{avg}}/0.4}$
implemented by Fast-NPS; a measured anisotropy must satisfy that model's
assumptions before it can be substituted for $r_{\mathrm{avg}}$ [2, 3].

In [ ]:
import math
import IMP
import IMP.bff as bff
import IMP.algebra

## 1. Direct, orientation-dependent FRET

Each `NPSDirectDye` stores $(x,y,z,m,\phi,r_{\mathrm{avg}})$ with
$m=-\cos\theta\in[-1,1]$ (not $+\cos\theta$) and $\phi$ in radians [2]. Let
$\mathbf u_D,\mathbf u_A$ be the mean transition-dipole directions,
$\hat{\mathbf r}$ the donor-to-acceptor direction,
$c_T=\mathbf u_D\cdot\mathbf u_A$ and
$c_i=\mathbf u_i\cdot\hat{\mathbf r}$. The Fast-NPS dynamic-regime expression
implemented in BFF is [2, 3]

$$\overline{\kappa^2}=q_Dq_A(c_T-3c_Dc_A)^2
+\frac{2-q_D-q_A}{3}+q_D(1-q_A)c_D^2+q_A(1-q_D)c_A^2,\qquad
q_i=\sqrt{r_{\mathrm{avg},i}/0.4}.$$

The **isotropically calibrated** radius needs a factor $3/2$:
$\overline R_0^{\,6}=(3/2)\overline{\kappa^2}R_{0,\mathrm{iso}}^6$ and
$E=\overline R_0^{\,6}/(\overline R_0^{\,6}+r^6)$ [2, 3].

In [ ]:
donor = bff.NPSDirectDye()
donor.x, donor.y, donor.z = 0.0, 0.0, 0.0
donor.m, donor.phi = 0.0, 0.0
donor.steady_state_anisotropy = 0.3  # synthetic input to Fast-NPS's q conversion

acceptor = bff.NPSDirectDye()
acceptor.x, acceptor.y, acceptor.z = 0.0, 0.0, 50.0
acceptor.m, acceptor.phi = 0.0, 0.0
acceptor.steady_state_anisotropy = 0.3

R0 = 50.0
eff = bff.nps_direct_fret_efficiency(donor, acceptor, R0)
# Both dipoles lie along x and the dye separation along z:
# c_T = 1 and c_D = c_A = 0, so the paper's equation has a simple oracle.
q = math.sqrt(0.3 / 0.4)
kappa2 = q * q + (2.0 - 2.0 * q) / 3.0
eff_analytic = (1.5 * kappa2) / (1.0 + 1.5 * kappa2)
assert math.isclose(eff, eff_analytic, rel_tol=1e-12)
print(f"oriented E = {eff:.4f} (analytic {eff_analytic:.4f})")

At $r=R_{0,\mathrm{iso}}$, $E=1/2$ **only** when
$\overline{\kappa^2}=2/3$. Setting both $r_{\mathrm{avg}}=0$ gives
$q_D=q_A=0$, removes angle dependence and recovers that isotropic limit;
$r_{\mathrm{avg}}=0.4$ instead gives $q=1$, the **rigid** limit in the
*direct* branch. `iso=True` alone does not change the direct FRET formula [2, 3].

In [ ]:
# The m coordinate is -cos(theta). A different m need not increase E:
# the relative geometry of both transition dipoles and the dye-pair axis matters.
for m_val in (0.0, 0.5, 0.9):
    d = bff.NPSDirectDye()
    d.x, d.y, d.z = 0.0, 0.0, 0.0
    d.m, d.phi = m_val, 0.0
    d.steady_state_anisotropy = 0.3
    print(f"donor m = {m_val:.1f}: E = "
          f"{bff.nps_direct_fret_efficiency(d, acceptor, R0):.4f}")

# Paper/implementation invariant: the fully depolarized direct model is angle-free.
donor.steady_state_anisotropy = acceptor.steady_state_anisotropy = 0.0
eff_iso = bff.nps_direct_fret_efficiency(donor, acceptor, R0)
assert math.isclose(eff_iso, 0.5, abs_tol=1e-12)
donor.steady_state_anisotropy = acceptor.steady_state_anisotropy = 0.3
print(f"fully depolarized E(R0_iso) = {eff_iso:.4f}")

## 2. Transfer anisotropy

The modeled transfer anisotropy is
$\hat r_T=q_Dq_A(3c_T^2-1)/5$ [2, 3]. For aligned mean dipoles with
$q_D=q_A=1$, the upper limit is $0.4$; a fully depolarized dye gives zero.
This observable contains dipole-orientation information; it must **not** be
silently added to a sampler that holds only $(x,y,z)$ coordinates.

In [ ]:
ta = bff.nps_direct_transfer_anisotropy(donor, acceptor)
assert math.isclose(ta, q * q * 2.0 / 5.0, rel_tol=1e-12)
print(f"transfer anisotropy: {ta:.4f}")

donor.steady_state_anisotropy = acceptor.steady_state_anisotropy = 0.4
ta_rigid = bff.nps_direct_transfer_anisotropy(donor, acceptor)
assert math.isclose(ta_rigid, 0.4, abs_tol=1e-12)
print(f"aligned rigid limit: {ta_rigid:.4f}")
donor.steady_state_anisotropy = acceptor.steady_state_anisotropy = 0.3

## 3. Multi-dye likelihood: synthetic observations

The direct network consumes one $(x,y,z,m,\phi)$ row and one `NPSNetworkDye`
per dye. Each measurement has a separate activity flag and standard deviation
for FRET and transfer anisotropy. The *illustrative* observations below are
chosen numbers, **not experimental results**. For independent Gaussian errors,
$\log L=-\sum_j[\frac12((y_j-\hat y_j)/\sigma_j)^2+
\log(\sqrt{2\pi}\sigma_j)]$; the normalization matters when comparing
models with different uncertainties [2, 3]. `r_iso6` is $R_{0,\mathrm{iso}}^6$,
not $R_{0,\mathrm{iso}}$. The dye orientation coordinates here are explicit;
this is an evaluator, not the position-only sampler in the next notebook.

In [ ]:
config = [
    [0.0, 0.0, 0.0, 0.2, 1.0],
    [30.0, 10.0, 20.0, -0.3, 0.4],
    [-10.0, 40.0, 15.0, 0.5, -0.7],
]

# Synthetic NPS steady-state-anisotropy inputs; direct branch with q > 0.
dyes = []
for anisotropy in (0.25, 0.30, 0.20):
    dye = bff.NPSNetworkDye()
    dye.dep = math.sqrt(anisotropy / 0.4)
    dye.iso, dye.dist_conv = False, False
    dyes.append(dye)

R_iso6 = 55.0 ** 6
eff01 = bff.nps_network_fret_efficiency(config, dyes, 0, 1, R_iso6)
ta01 = bff.nps_network_transfer_anisotropy(config, dyes, 0, 1)
print(f"network E(0,1) = {eff01:.4f}, transfer anisotropy = {ta01:.4f}")

In [ ]:
def make_measurement(dye1, dye2, fret=False, e_avg=0.0, e_err=1.0,
                     r_iso6=0.0, ta=False, r_t_avg=0.0, r_t_err=1.0):
    measurement = bff.NPSMeasurement()
    measurement.dye1, measurement.dye2 = dye1, dye2
    measurement.fret_active, measurement.e_avg, measurement.e_err = fret, e_avg, e_err
    measurement.r_iso6 = r_iso6
    measurement.ta_active, measurement.r_t_avg, measurement.r_t_err = ta, r_t_avg, r_t_err
    return measurement

measurements = [
    make_measurement(0, 1, fret=True, e_avg=0.60, e_err=0.08, r_iso6=55.0**6),
    make_measurement(0, 2, fret=True, e_avg=0.35, e_err=0.10, r_iso6=50.0**6),
    make_measurement(1, 2, fret=True, e_avg=0.50, e_err=0.12, r_iso6=60.0**6),
    make_measurement(0, 1, ta=True, r_t_avg=0.05, r_t_err=0.02),
]

ll = bff.nps_network_log_likelihood(config, dyes, measurements)
# Independently assemble each normalized Gaussian term, keeping FRET and
# transfer-anisotropy errors and predictions distinct.
terms = []
for m in measurements:
    if m.fret_active:
        pred = bff.nps_network_fret_efficiency(config, dyes, m.dye1, m.dye2, m.r_iso6)
        terms.append(-0.5 * ((m.e_avg - pred) / m.e_err) ** 2
                     - math.log(math.sqrt(2 * math.pi) * m.e_err))
    if m.ta_active:
        pred = bff.nps_network_transfer_anisotropy(config, dyes, m.dye1, m.dye2)
        terms.append(-0.5 * ((m.r_t_avg - pred) / m.r_t_err) ** 2
                     - math.log(math.sqrt(2 * math.pi) * m.r_t_err))
assert len(terms) == 4
assert math.isclose(ll, sum(terms), rel_tol=1e-12, abs_tol=1e-12)
print(f"four-observable normalized log L = {ll:.4f}")

## 4. Position-cloud convolution (a different forward model)

The *direct* model above evaluates one dye-pair separation. A convolved
measurement instead averages the nonlinear $E(r)$ over two local dye-position
clouds. That average is **not** generally equal to $E$ at the mean dye positions
[1, 3, 4]. BFF generates a degree-11 power-basis approximation on $d=1..150$ Å
and selects (does not interpolate between) its orientation row before
polynomial evaluation. A fully isotropic pair uses row 0; mixed and two-grid
pairs need 25 and 625 rows respectively [3].

**API nuance:** in `nps_convolved_efficiency_table` a `dep` argument of `1.0`
selects the isotropically averaged table *layout*. It is not a claim that
`NPSNetworkDye.dep=1.0` means isotropic in the direct branch (there $q=1$
is rigid). The evaluator's `iso`/`dist_conv` flags select table rows; this
example keeps both dyes fully depolarized for consistent metadata.

In [ ]:
# Five local dye offsets (same cloud shape for both labels); the site
# separation is added by the BFF table generator's distance axis.
offsets = [(0.0, 0.0, 0.0), (2.0, 0.0, 0.0), (-2.0, 0.0, 0.0),
           (0.0, 2.0, 0.0), (0.0, -2.0, 0.0)]
av = [v for pt in offsets for v in (*pt, 0.2)]  # flat x,y,z,weight
coeff = bff.nps_convolved_efficiency_table(
    av, av, 1.0, 1.0, R_iso6, n_samples=6000, seed=7)
assert len(coeff) == 1 and len(coeff[0]) == 12

iso_dyes = []
for _ in range(2):
    dye = bff.NPSNetworkDye()
    dye.dep, dye.iso, dye.dist_conv = 0.0, True, True
    iso_dyes.append(dye)
distance01 = math.dist(config[0][:3], config[1][:3])
e_conv = bff.nps_network_fret_efficiency(config, iso_dyes, 0, 1, R_iso6, coeff)
e_row0 = bff.nps_convolved_efficiency(distance01, coeff[0])
assert math.isclose(e_conv, e_row0, rel_tol=1e-12)
print(f"site distance {distance01:.1f} Å: convolved E = {e_conv:.4f} (row 0)")

## 5. Standalone isotropic direct-point likelihood

`nps_isotropic_direct_score` uses
$\hat E=1/(1+(r/R_{0,\mathrm{iso}})^6)$ and returns the *negative normalized*
Gaussian log likelihood of $E_{\mathrm{obs}}$ given $\hat E+b_E$ and standard
deviation $\sigma_E$. `log_r_iso` is $\log(R_{0,\mathrm{iso}}/1\,\text{Å})$.
This simple point-position score is **not** the cloud-convolved likelihood,
and an additive efficiency bias must be calibrated or given a proper prior
before inference [2, 3]. A caller may compose this scalar into a suitable
objective; it is not automatically plugged into `NPSNetworkObjective`.

In [ ]:
point_donor = [0.0, 0.0, 0.0]
log_r_iso = math.log(55.0)
for r in (48.0, 55.0, 62.0):
    score = bff.nps_isotropic_direct_score(
        point_donor, [0.0, 0.0, r], 0.0, log_r_iso,
        0.55, 0.08)
    predicted = 1.0 / (1.0 + (r / 55.0) ** 6)
    gaussian_score = (0.5 * ((0.55 - predicted) / 0.08) ** 2
                      + math.log(0.08 * math.sqrt(2.0 * math.pi)))
    assert math.isclose(score, gaussian_score, rel_tol=1e-12, abs_tol=1e-12)
    print(f"r = {r:.0f} Å, E = {predicted:.4f}, -log L = {score:.4f}")

## Primary references and provenance

[1] Muschielok A *et al.* **A nano-positioning system for macromolecular
structural analysis.** *Nature Methods* **5**, 965–971 (2008).
doi:10.1038/nmeth.1259.

[2] Muschielok A & Michaelis J. **Application of the Nano-Positioning System
to the Analysis of Fluorescence Resonance Energy Transfer Networks.**
*J. Phys. Chem. B* **115**, 11927–11937 (2011). doi:10.1021/jp2060377.
Their supporting theory gives the invariant orientation coordinate
$m=-\cos\theta$ and its prior; the code assertions above check the
Fast-NPS-convention forward model rather than fitting any published dataset.

[3] Eilert T, Beckers M, Drechsler F & Michaelis J. **Fast-NPS—A Markov Chain
Monte Carlo-based analysis tool to obtain structural information from
single-molecule FRET measurements.** *Comput. Phys. Commun.* **219**,
377–389 (2017). doi:10.1016/j.cpc.2017.05.027. The released Fast-NPS code
is the behavioral comparator for BFF's branch/table layout; BFF's cloud
sampler and polynomial fit are documented modifications, not exact code parity.

[4] Beckers M *et al.* **Quantitative structural information from
single-molecule FRET.** *Faraday Discuss.* **184**, 117–129 (2015).
doi:10.1039/c5fd00110b.